# 003: Bag of Words and TF-IDF — Vector space model and information-theoretic weighting

## TF-IDF DERIVATION
- **Term Frequency**: $\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total terms in } d}$
- **Inverse Document Frequency** (information theory view): Rare terms carry more information:
  $$\text{IDF}(t) = \log\frac{N}{|\{d : t \in d\}|}$$
- **TF-IDF**: $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$
---


In [ ]:
import numpy as np
from collections import Counter
import math

class TFIDFVectorizer:
    """From-scratch TF-IDF implementation."""
    def __init__(self):
        self.vocab = {}
        self.idf = {}

    def fit(self, documents):
        all_words = set()
        for doc in documents:
            all_words.update(doc.split())
        self.vocab = {w: i for i, w in enumerate(sorted(all_words))}
        N = len(documents)
        for word in self.vocab:
            df = sum(1 for doc in documents if word in doc.split())
            self.idf[word] = math.log(N / (df + 1e-10))

    def transform(self, documents):
        matrix = np.zeros((len(documents), len(self.vocab)))
        for i, doc in enumerate(documents):
            words = doc.split()
            counts = Counter(words)
            total = len(words)
            for word, count in counts.items():
                if word in self.vocab:
                    tf = count / total
                    matrix[i, self.vocab[word]] = tf * self.idf[word]
        return matrix



In [ ]:
docs = ["the cat sat on the mat", "the dog sat on the log", "cats and dogs are friends"]
vectorizer = TFIDFVectorizer()
vectorizer.fit(docs)
tfidf_matrix = vectorizer.transform(docs)

print("--- TF-IDF Matrix ---")
print(f"Vocabulary: {list(vectorizer.vocab.keys())}")
for i, doc in enumerate(docs):
    top_idx = np.argsort(tfidf_matrix[i])[-3:][::-1]
    top_words = [(list(vectorizer.vocab.keys())[j], tfidf_matrix[i, j]) for j in top_idx]
    print(f"Doc {i}: Top terms = {top_words}")

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

print(f"\nCosine(doc0, doc1) = {cosine_sim(tfidf_matrix[0], tfidf_matrix[1]):.4f}")
print(f"Cosine(doc0, doc2) = {cosine_sim(tfidf_matrix[0], tfidf_matrix[2]):.4f}")
